### Calculus and Differentiation for Neural Networks

The math underneath every gradient computed by hand across this series, and what `.backward()` in `pytorch.ipynb` automates. Derivatives, partial derivatives, the chain rule (the single most important rule here), gradients, Jacobian, Hessian (ties directly to `boosting.ipynb`'s XGBoost Hessian usage).

#### 0. Derivatives: rate of change

Definition: f'(x) = lim(h->0) [f(x+h) - f(x)] / h, the instantaneous rate of change of f at x, the slope of the tangent line at that exact point.

Worked example, f(x) = x^2, at x=3, approximated with a small but nonzero h=0.001 (this is literally what the limit definition converges to as h shrinks toward 0):
```
f(3) = 9
f(3.001) = 3.001^2 = 9.006001

f'(3) ~ (9.006001 - 9) / 0.001 = 0.006001/0.001 = 6.001
```
The exact analytical derivative of x^2 is 2x, at x=3 that is exactly 6, the tiny-h approximation (6.001) confirms it. This numerical approximation is also how gradient-checking works in practice, verifying an analytically-derived (or autograd-computed) gradient by comparing it against this tiny-step finite-difference estimate.

In [ ]:
def f(x):
    return x ** 2

x, h = 3.0, 0.001
numerical_derivative = (f(x + h) - f(x)) / h
analytical_derivative = 2 * x  # d/dx(x^2) = 2x

print("numerical (finite difference):", numerical_derivative)
print("analytical (2x):", analytical_derivative)

#### 1. Partial derivatives

For a function of MULTIPLE variables, the partial derivative with respect to one variable holds every other variable fixed and measures the rate of change along just that one direction.

Worked example, f(w,x) = w*x + w^2 (a toy loss-like function of a weight w and an input x), at w=2, x=3:
```
df/dw = x + 2w    (treating x as constant)   = 3 + 2*2 = 7
df/dx = w         (treating w as constant)   = 2
```
This is exactly how `dL/dw` is computed throughout this series while `dL/db` is computed separately, each partial derivative asks "if I nudge ONLY this one parameter, holding everything else fixed, how much does the output change."

In [ ]:
import sympy as sp

w, x = sp.symbols("w x")
f_expr = w * x + w**2

df_dw = sp.diff(f_expr, w)
df_dx = sp.diff(f_expr, x)

print("df/dw =", df_dw, "-> at w=2,x=3:", df_dw.subs({w: 2, x: 3}))
print("df/dx =", df_dx, "-> at w=2,x=3:", df_dx.subs({w: 2, x: 3}))

#### 2. The chain rule, worked exactly as the logreg gradient derivation used it

Formula: if y = f(g(x)), then dy/dx = df/dg * dg/dx, differentiate the outer function, then multiply by the derivative of the inner function. This is THE rule that makes training any layered model (logistic regression's loss-through-sigmoid, or a full neural network's loss-through-many-layers) possible at all.

Worked, reproducing `classical-ml.ipynb`'s from-scratch logreg derivation step by step: L depends on p, p depends on z, z depends on w. Three nested functions, chain rule links them:
```
dL/dw = dL/dp * dp/dz * dz/dw
```
At x=2, w=0.5, b=0.1, y=1 (the exact setup used in `pytorch.ipynb`'s autograd verification):
```
z = 1.1, p = sigmoid(1.1) = 0.7503

dL/dp = (p-y)/(p*(1-p))   [derivative of -[y*ln(p)+(1-y)*ln(1-p)] w.r.t. p]
      = (0.7503-1)/(0.7503*0.2497) = -0.2497/0.1873 = -1.333
dp/dz = p*(1-p) = 0.7503*0.2497 = 0.1873          [sigmoid's own derivative]
dz/dw = x = 2                                       [z=w*x+b, so dz/dw=x]

dL/dw = (-1.333) * (0.1873) * (2) = -0.4994
```
The (p-y)/(p*(1-p)) and p*(1-p) terms cancel exactly, leaving the clean (p-y)*x result derived directly in `classical-ml.ipynb`, and confirmed numerically identical there via PyTorch autograd. Three chain-rule steps multiplied together, that IS backpropagation for this one-layer case, a full neural network just chains many more of these steps, one per layer.

In [ ]:
x_val, w_val, b_val, y_val = 2.0, 0.5, 0.1, 1.0

z = w_val * x_val + b_val
p = 1 / (1 + (2.71828 ** -z))

dL_dp = (p - y_val) / (p * (1 - p))
dp_dz = p * (1 - p)
dz_dw = x_val

dL_dw_chain = dL_dp * dp_dz * dz_dw
dL_dw_simplified = (p - y_val) * x_val

print("chain rule, 3 separate terms multiplied:", dL_dw_chain)
print("simplified closed form (p-y)*x:", dL_dw_simplified)
print("match:", round(dL_dw_chain, 4) == round(dL_dw_simplified, 4))

#### 3. Gradient: the vector of all partial derivatives

The gradient (written grad-f or nabla-f) is just every partial derivative stacked into one vector, [df/dw1, df/dw2, ..., df/dwn]. "Gradient descent" (used in every `for i in range(n_iters)` loop across this series) means stepping in the direction OPPOSITE this vector, since the gradient points in the direction of steepest INCREASE, stepping opposite it decreases the loss fastest.

#### 4. Jacobian and Hessian

Jacobian: the matrix of partial derivatives for a VECTOR-valued function (multiple outputs, not just one), each row is the gradient of one output with respect to every input. Multi-class logreg's softmax output is exactly this case, K outputs, the Jacobian describes how every output probability changes with respect to every input logit simultaneously.

Hessian: the matrix of SECOND derivatives, how the gradient itself is changing, curvature information. This is exactly the H in `boosting.ipynb`'s XGBoost derivation, h=p(1-p), XGBoost uses this second-derivative (curvature) information directly in its leaf-weight and Gain formulas, which is precisely why it converges faster than classic GBM's first-order-only approach (also covered in `boosting.ipynb`).

In [ ]:
import torch

w = torch.tensor(0.5, requires_grad=True)
x, b, y = torch.tensor(2.0), torch.tensor(0.1), torch.tensor(1.0)

z = w * x + b
p = torch.sigmoid(z)
loss = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))

grad = torch.autograd.grad(loss, w, create_graph=True)[0]
hessian = torch.autograd.grad(grad, w)[0]

print("gradient dL/dw:", grad.item())
print("second derivative (Hessian, 1x1 here) d2L/dw2:", hessian.item())

#### 5. Backpropagation is just the chain rule, applied layer by layer

A deep network is a long chain of nested functions, loss depends on the last layer's output, which depends on the second-to-last layer's output, and so on back to the input. Backpropagation computes the gradient with respect to EVERY layer's weights by applying the chain rule repeatedly, working backward from the loss, reusing each layer's local gradient as it goes, exactly the same 3-step chain multiplication worked out by hand above, just repeated once per layer instead of once. This is precisely what `.backward()` automates in `pytorch.ipynb`, it never derives a closed-form formula the way the hand-worked examples in this series did, it mechanically applies the chain rule at every operation that was run, in reverse order.